# SangHyo CN / MCI / DEM 자동 실행 노트북

팀 공통 `base.ipynb`와 같은 실행 골조입니다. Colab에서 실행할 때마다 GitHub 저장소를 새로 clone하므로, 코드를 push한 뒤 **런타임 → 모두 실행**하면 최신 `Transformer + TabNet + Google YDF` 실험이 실행됩니다.

- GitHub의 최신 코드를 `/content/Google-Ajou-AICapstone`에 새로 받습니다.
- 공용 Google Drive의 `Data`를 읽습니다.
- Training-only EDA를 먼저 실행한 뒤 학습을 시작합니다.
- 모든 결과와 ZIP 압축본은 개인 Google Drive에 저장합니다.


## 셀 1. 기본 환경 준비

Colab에서는 Google Drive를 마운트하고 기존 임시 저장소를 삭제한 뒤 GitHub에서 최신 코드를 clone합니다. 로컬에서는 현재 저장소를 그대로 사용합니다.


In [ ]:
# Cell 1 - Environment setup
import json
import os
import runpy
import shutil
import subprocess
import sys
import traceback
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/Pig30nidaE/Google-Ajou-AICapstone.git"
REPO_DIR_NAME = "Google-Ajou-AICapstone"


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / "base.ipynb").exists() or (path / "Data").exists():
            return path
    return None


IN_COLAB = in_colab()

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    clone_path = Path("/content") / REPO_DIR_NAME
    os.chdir("/content")
    if clone_path.exists():
        print(f"Remove existing repo: {clone_path}")
        shutil.rmtree(clone_path)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(clone_path)],
        check=True,
    )
    PROJECT_ROOT = clone_path.resolve()
else:
    PROJECT_ROOT = find_project_root() or Path.cwd().resolve()

os.chdir(PROJECT_ROOT)

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/MyDrive/GoogleAI_contest/Data")
else:
    DATA_ROOT = PROJECT_ROOT / "Data"

if not DATA_ROOT.exists() and (PROJECT_ROOT / "Data").exists():
    DATA_ROOT = PROJECT_ROOT / "Data"

print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_ROOT    : {DATA_ROOT}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Remove existing repo: /content/Google-Ajou-AICapstone
IN_COLAB     : True
PROJECT_ROOT : /content/Google-Ajou-AICapstone
DATA_ROOT    : /content/drive/MyDrive/GoogleAI_contest/Data


## 셀 2. 사용자 입력

일반적으로 `RUN_MODE`만 `smoke` 또는 `full`로 선택하면 됩니다. 정식 결과는 `full`입니다.


In [ ]:
# Cell 2 - User inputs
USER_FOLDER = "SangHyo"
EXPERIMENT_FOLDER = "ThreeClass_TransformerTabNet_Google"
RUN_FILE = "train.py"
EDA_FILE = "eda.py"

RUN_MODE = "full"  # 빠른 동작 확인: "smoke", 정식 실행: "full"
FEATURE_MODE = "clinical_plus_lifelog"  # 비교용: "wearable_only"
SEED = 20260719

# None이면 Colab은 MyDrive/SangHyo_CN_MCI_DEM_Results에 저장합니다.
RESULTS_ROOT_OVERRIDE = None


## 셀 3. 경로 확인

GitHub에서 받은 실험 코드, 공용 데이터, Google Drive 결과 경로를 확인합니다. 하나라도 없으면 학습 전에 바로 중단합니다.


In [ ]:
# Cell 3 - Resolve paths
USER_ROOT = (PROJECT_ROOT / USER_FOLDER).resolve()
EXPERIMENT_ROOT = (USER_ROOT / EXPERIMENT_FOLDER).resolve()
RUN_PATH = (EXPERIMENT_ROOT / RUN_FILE).resolve()
EDA_PATH = (EXPERIMENT_ROOT / EDA_FILE).resolve()
REQUIREMENTS_PATH = (EXPERIMENT_ROOT / "requirements_colab.txt").resolve()
TRAINING_ROOT = (DATA_ROOT / "1.Training").resolve()
VALIDATION_ROOT = (DATA_ROOT / "2.Validation").resolve()

if RESULTS_ROOT_OVERRIDE is not None:
    RESULTS_ROOT = Path(RESULTS_ROOT_OVERRIDE).expanduser().resolve()
elif IN_COLAB:
    RESULTS_ROOT = Path("/content/drive/MyDrive/SangHyo_CN_MCI_DEM_Results")
else:
    RESULTS_ROOT = EXPERIMENT_ROOT / "training_outputs"

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
if FEATURE_MODE not in {"clinical_plus_lifelog", "wearable_only"}:
    raise ValueError("Unknown FEATURE_MODE.")

required_paths = {
    "user folder": USER_ROOT,
    "experiment folder": EXPERIMENT_ROOT,
    "training script": RUN_PATH,
    "EDA script": EDA_PATH,
    "requirements": REQUIREMENTS_PATH,
    "training data": TRAINING_ROOT,
    "validation data": VALIDATION_ROOT,
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing))

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = RESULTS_ROOT / f"{RUN_ID}_{RUN_MODE}_{FEATURE_MODE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

for name, path in required_paths.items():
    print(f"{name:20s}: {path}")
print(f"{'output':20s}: {OUTPUT_DIR}")


user folder         : /content/Google-Ajou-AICapstone/SangHyo
experiment folder   : /content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google
training script     : /content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/train.py
EDA script          : /content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/eda.py
requirements        : /content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/requirements_colab.txt
training data       : /content/drive/MyDrive/GoogleAI_contest/Data/1.Training
validation data     : /content/drive/MyDrive/GoogleAI_contest/Data/2.Validation
output              : /content/drive/MyDrive/SangHyo_CN_MCI_DEM_Results/20260720_005257_full_clinical_plus_lifelog


## 셀 4. 실험 requirements 설치

실험 폴더의 `requirements_colab.txt`를 설치합니다. Colab에 포함된 CUDA용 PyTorch는 다시 설치하지 않습니다.


In [ ]:
# Cell 4 - Install experiment requirements
print(f"Installing: {REQUIREMENTS_PATH}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REQUIREMENTS_PATH),
    ],
    check=True,
)
print("Requirements installation complete.")


Installing: /content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/requirements_colab.txt
Requirements installation complete.


## 셀 5. EDA와 학습 자동 실행

A100을 확인하고 Training-only EDA를 실행한 다음, Transformer·TabNet·Google YDF nested-CV 학습을 시작합니다. 끝나면 핵심 지표를 표시하고 전체 결과를 Google Drive ZIP으로 저장합니다.


In [ ]:
# Cell 5 - Run EDA and training
from importlib.metadata import version

import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU가 없습니다. Colab 런타임 유형에서 A100 GPU를 선택해주세요."
    )

print(f"GPU            : {torch.cuda.get_device_name(0)}")
print(f"CUDA           : {torch.version.cuda}")
print(f"pytorch-tabnet : {version('pytorch-tabnet')}")
print(f"YDF            : {version('ydf')}")
print(f"Optuna         : {version('optuna')}")

EDA_OUTPUT_DIR = OUTPUT_DIR / "eda"
TRAINING_OUTPUT_DIR = OUTPUT_DIR / "training"

eda_command = [
    sys.executable,
    str(EDA_PATH),
    "--training-root",
    str(TRAINING_ROOT),
    "--output-dir",
    str(EDA_OUTPUT_DIR),
]

training_command = [
    sys.executable,
    str(RUN_PATH),
    "--training-root",
    str(TRAINING_ROOT),
    "--validation-root",
    str(VALIDATION_ROOT),
    "--output-dir",
    str(TRAINING_OUTPUT_DIR),
    "--feature-mode",
    FEATURE_MODE,
    "--outer-folds",
    "3",
    "--outer-repeats",
    "2",
    "--inner-folds",
    "3",
    "--trials-transformer",
    "24",
    "--trials-tabnet",
    "24",
    "--trials-ydf",
    "40",
    "--seed",
    str(SEED),
]
if RUN_MODE == "smoke":
    training_command.append("--fast")

def run_python_file(script_path, arguments):
    """공통 base.ipynb처럼 현재 Colab 커널에서 파일을 실행합니다."""
    previous_cwd = Path.cwd()
    previous_argv = sys.argv[:]
    script_dir = str(script_path.parent)
    inserted_path = script_dir not in sys.path
    if inserted_path:
        sys.path.insert(0, script_dir)
    os.chdir(script_path.parent)
    sys.argv = [str(script_path), *arguments]
    try:
        return runpy.run_path(str(script_path), run_name="__main__")
    finally:
        sys.argv = previous_argv
        os.chdir(previous_cwd)
        if inserted_path and script_dir in sys.path:
            sys.path.remove(script_dir)


try:
    print("\n[1/2] Training-only EDA")
    print(" ".join(eda_command))
    run_python_file(EDA_PATH, eda_command[2:])

    print("\n[2/2] Model training")
    print(" ".join(training_command))
    run_python_file(RUN_PATH, training_command[2:])
except BaseException:
    failure_text = traceback.format_exc()
    failure_path = OUTPUT_DIR / "FAILED_TRACEBACK.log"
    failure_path.write_text(failure_text, encoding="utf-8")
    print("\n실제 오류 traceback:\n")
    print(failure_text)
    print(f"오류 로그 저장: {failure_path}")
    raise

report_path = TRAINING_OUTPUT_DIR / "FINAL_REPORT.json"
with report_path.open(encoding="utf-8") as handle:
    report = json.load(handle)

nested = report["nested_oof"]
validation = report.get("validation")
metric_names = ["accuracy", "macro_f1", "roc_auc_ovr_macro", "balanced_accuracy"]
summary_rows = [
    {"split": "nested OOF", **{name: nested[name] for name in metric_names}}
]
if validation is not None:
    summary_rows.append(
        {"split": "validation", **{name: validation[name] for name in metric_names}}
    )
display(pd.DataFrame(summary_rows))
print("Target check:", report["target_check_nested_oof"])

archive_base = RESULTS_ROOT / f"{OUTPUT_DIR.name}_archive"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
print(f"Result folder : {OUTPUT_DIR}")
print(f"Result archive: {archive_path}")
print("Done.")


GPU            : NVIDIA A100-SXM4-40GB
CUDA           : 12.8
pytorch-tabnet : 4.1.0
YDF            : 0.16.1
Optuna         : 4.9.0

[1/2] Training-only EDA
/usr/bin/python3 /content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/eda.py --training-root /content/drive/MyDrive/GoogleAI_contest/Data/1.Training --output-dir /content/drive/MyDrive/SangHyo_CN_MCI_DEM_Results/20260720_005257_full_clinical_plus_lifelog/eda
# CN / MCI / DEM Training EDA 보고서

이 보고서는 **학습 데이터만** 살펴본 결과입니다. 모델 학습이나 Validation 라벨 확인은 하지 않았습니다.

## 먼저 알아둘 점

- 사람 수는 CN 85명, MCI 47명, DEM 9명입니다.
- Gait/Sleep/CognitiveFunction의 라벨 사본 3개는 사람과 정답이 모두 같습니다.
- DEM은 9명뿐이라 한두 명의 결과가 점수를 크게 바꿉니다. 그래서 단일 accuracy보다 Macro F1과 class별 결과를 함께 봐야 합니다.
- Activity와 Sleep은 각각 9,705행이며, 두 자료가 같은 사람·날짜에 맞는 경우는 9,673건입니다.
- 가공 후 사람당 입력 후보는 754개입니다. 실제 학습에서는 각 fold의 Training 부분 안에서만 줄입니다.

## 눈에 띄는 패턴

### 1. 인지검사 점수는 강하지만, 정답 열은 반드시 제외해야 합니다

MMSE TOTAL 중앙값은 CN 28.0, MCI 26.0, DEM 22.0입니다. 
다만 MMSE 파일의 `DIAG_NM`은 이번 정답

/content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/train.py:234: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(seed=seed, multivariate=True)
[I 2026-07-20 00:54:49,822] A new study created in memory with name: no-name-412e20ff-b6a8-4d77-8e2e-9c04f28821fa


  0%|          | 0/24 [00:00<?, ?it/s]

/content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/models.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_layers)
/content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/models.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_layers)
/content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/models.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_layers)


[I 2026-07-20 00:55:22,047] Trial 0 finished with value: 0.42882725060393234 and parameters: {'max_features': 96, 'd_token': 64, 'n_heads': 4, 'n_layers': 4, 'ff_multiplier': 2.0, 'dropout': 0.1206739694646371, 'lr': 0.0008269739695621385, 'weight_decay': 0.006129732864535711, 'epochs': 420, 'batch_size': 64, 'label_smoothing': 0.04460454259255053, 'noise_std': 0.018043272370630838, 'class_weight_power': 0.589792682064892}. Best is trial 0 with value: 0.42882725060393234.


...

Train model on 62 examples
Model trained in 0:00:00.524064
Train model on 63 examples
Model trained in 0:00:00.518554
Train model on 63 examples
Model trained in 0:00:00.542347
[I 2026-07-20 05:19:48,842] Trial 0 finished with value: 0.46594367475821 and parameters: {'max_features': 128, 'num_trees': 600, 'max_depth': 2, 'min_examples': 14, 'shrinkage': 0.02, 'subsample': 0.85, 'use_hessian_gain': False, 'l2_regularization': 0.0021839915040766517, 'class_weight_power': 0.503708826783254}. Best is trial 0 with value: 0.46594367475821.
Train model on 62 examples
Model trained in 0:00:10.673051
Train model on 63 examples
Model trained in 0:00:10.597592
Train model on 63 examples
Model trained in 0:00:10.758288
[I 2026-07-20 05:20:39,346] Trial 1 finished with value: 0.36088128676546494 and parameters: {'max_features': 192, 'num_trees': 1600, 'max_depth': 4, 'min_examples': 4, 'shrinkage': 0.1, 'subsample': 0.85, 'use_hessian_gain': False, 'l2_regularization': 0.0013010571026357655, 'class

/content/Google-Ajou-AICapstone/SangHyo/ThreeClass_TransformerTabNet_Google/models.py:92: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=cfg.n_layers)
/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


Successfully saved model at /content/drive/MyDrive/SangHyo_CN_MCI_DEM_Results/20260720_005257_full_clinical_plus_lifelog/training/models/repeat_01_fold_02/tabnet/model.zip
Train model on 94 examples
Model trained in 0:00:04.381094


,split,accuracy,macro_f1,roc_auc_ovr_macro,balanced_accuracy
0,nested OOF,0.609929,0.491265,0.706443,0.477819
1,validation,0.848485,0.636364,0.727896,0.653846


Target check: {'thresholds': {'accuracy': 0.8, 'macro_f1': 0.7, 'roc_auc_ovr_macro': 0.85}, 'checks': {'accuracy': False, 'macro_f1': False, 'roc_auc_ovr_macro': False}, 'all_targets_met': False, 'note': '목표값은 희망 기준이며 데이터가 보장하는 값이 아닙니다.'}
Result folder : /content/drive/MyDrive/SangHyo_CN_MCI_DEM_Results/20260720_005257_full_clinical_plus_lifelog
Result archive: /content/drive/MyDrive/SangHyo_CN_MCI_DEM_Results/20260720_005257_full_clinical_plus_lifelog_archive.zip
Done.
